# DeepRL Monopoly — 1v1 self-play vs ASU

Target metric: win rate specifically against ASU. 1v1 baseline is 50% (vs
25% in a 4-seat table where you must beat Builder AND DealMaker too), so
this is the fastest realistic path to a high win-rate number today.

Still pure self-play RL — ASU is only ever called as a black-box opponent
via `choose_action(env)`. No imitation loss, no reading ASU's outputs as
training labels, no distillation.

## 1. Mount Drive (own checkpoint folder)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/DeepRL_Monopoly_ckpt_vs_asu'
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 2. Clone the repo (feature/asu-teacher-distillation branch)

In [ ]:
%cd /content
!rm -rf DeepRL_Monopoly
!git clone --branch feature/asu-teacher-distillation https://github.com/EnzeCbe/monopoly-boom.git DeepRL_Monopoly
%cd DeepRL_Monopoly

## 3. Check GPU + torch

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 4. Train 1v1 vs ASU

This script isn't GPU-accelerated at the env-simulation level (Monopoly's
step logic is CPU/Python), so GPU mainly speeds up the PPO network update,
not the per-decision loop. Still worth running here for extra parallel
throughput alongside the local run.

In [ ]:
import os
OUT = f"{CHECKPOINT_DIR}/vs_asu_1v1.pt"
os.makedirs(os.path.dirname(OUT), exist_ok=True)

!PYTHONIOENCODING=utf-8 python tools/train_vs_asu_1v1.py \
  --algo ppo --games 5000 --seed 11 \
  --checkpoint-every 50 --log-every 25 \
  --out "{OUT}"

## 5. Resume after a disconnect

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/train_vs_asu_1v1.py \
  --algo ppo --games 20000 --seed 11 --resume \
  --checkpoint-every 50 --log-every 25 \
  --out "{OUT}"

## 6. Analyze the per-game log

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/analyze_run.py "{OUT.rsplit('.', 1)[0]}_games.csv" --window 50